# NB1 · Setup — run this once, and read every output

**Nothing else in this project will work until this notebook ends with `GO`.**

It does five things, in an order chosen so the cheap checks fail before the
expensive ones start:

| step | cost | what it catches |
|---|---|---|
| 1. environment | seconds | a CPU-only torch, a missing package |
| 2. offline proof | seconds | a dependency that fetches on first use |
| 3. pack the dataset | 20–40 min | a source tree that differs from the one this port was designed against |
| 4. preflight the zoo | ~5 min GPU | an architecture that cannot run at a resolution the oracle will sweep |
| 5. kill-and-resume | ~5 min GPU | the defect class that has cost this project five separate times |

## Why the preflight matters more than it looks

On CIFAR-100 this step caught three defects **before a single GPU-hour was
spent**: a ViT whose positional embedding was sized for one patch grid, an
MLP-Mixer that cannot run at any other resolution at all, and a budget table
with duplicate entries that would have made MSC mathematically undefined
three hours into a training phase.

The eight architectures here are new and four of them are decomposed from
torchvision. **That decomposition is the part most likely to be wrong**, and
this is where it surfaces.

## What "GO" means

Every architecture builds, forwards, backprops, exits at every stage, prices a
strictly-ascending budget table, and survives a real interrupt. Anything less
and the numbers this project produces would be well-formed and meaningless.

In [ ]:
# ============================================================================
# CELL 1 -- unpack the library.  Runs in every notebook.  No network.
# ============================================================================
# Writes two files into the working directory and imports them:
#
#   msc_lib.py    28c29d2d3a74   the pipeline: data, zoo, training, measurement
#   msc_core.py   2cc4ba5e0935   the reference maths: the MSC definition and
#                                    every statistic in the paper
#
# Both are GENERATED from src/ by build_notebooks_in100.py. Editing the base64
# below does nothing that survives a rebuild -- edit src/msc_lib.py instead.
#
# NOTHING IS INSTALLED HERE. This pipeline runs offline; the packages must
# already be present (see requirements.txt). A missing one is reported by name
# with what it costs you, rather than silently pip-installing on a machine that
# may have no network.
import base64, os, sys
from pathlib import Path

# Offline guards must be set BEFORE anything that might fetch is imported.
os.environ.setdefault('MSC_OFFLINE', '1')

WORK = Path.cwd()
_LIB = (
    '...',
    'b3IgdGhlIG9mZmxpbmUgY2hlY2tzIikK',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]          # force reimport if this cell is re-run

_MISSING = []
for _pkg, _why in (('torch', 'everything'),
                   ('torchvision', 'resnet/vgg/shufflenet/swin'),
                   ('numpy', 'everything'), ('pandas', 'every table'),
                   ('pyarrow', 'per_sample/*.parquet -- the science'),
                   ('yaml', 'config.yaml per run'),
                   ('scipy', 'Spearman = Q1 and Q3'),
                   ('sklearn', 'Q4 delta-R2, Q2 PCA'),
                   ('psutil', 'host telemetry columns'),
                   ('pynvml', 'GPU power -- energy columns are NA without it'),
                   ('fvcore', 'FLOPs. rho is DEFINED in FLOPs.')):
    try:
        __import__(_pkg)
    except ImportError:
        _MISSING.append(f'{_pkg:12s} {_why}')
if _MISSING:
    print('MISSING PACKAGES -- install these, then restart the kernel:')
    for _m in _MISSING:
        print('   ', _m)
    raise SystemExit('see requirements.txt')

import msc_lib as M
import torch

print(f'msc_lib {M.__version__}   torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  GPU {_i}: {_p.name}  {_p.total_memory/2**30:.1f} GiB  sm_{_p.major}{_p.minor}')
else:
    print('  *** NO CUDA. A CPU-only torch trains at roughly 1/200th speed')
    print('  *** while reporting entirely plausible numbers. Fix this first.')

---
## Step 1 · Where everything lives

Set the two paths below. `DATA_DIR` needs ~24 GiB, `MSC_ROOT` grows to ~60–90
GiB across the full programme.

In [ ]:
# ============================================================================
# CELL 2 -- WHERE EVERYTHING LIVES
# ============================================================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine gets `msc_data/in100` and `msc_results`.
#
# The previous version defaulted to r'D:\msc_data\in100'. There is no D:
# drive here, and the failure was
#
#     FileNotFoundError: [WinError 3] The system cannot find the path
#     specified: 'D:\'
#
# forty lines deep inside pathlib, naming neither the setting nor the file that
# had to change. A default that names a drive letter is wrong on any machine
# without that letter (D-44).
#
# Set them explicitly if you want somewhere specific. Both are checked below by
# WRITING A PROBE FILE AND READING IT BACK -- os.access lies on Windows shares.
#
#   data     ~26 GB   the packed dataset, read-only after NB1
#   results ~120 GB   every run. Nothing here is ever deleted.

DATA_DIR = None      # e.g. r'E:\msc_data\in100'   -- None = choose for me
MSC_ROOT = None      # e.g. r'E:\msc_results'        -- None = choose for me

# ---------------------------------------------------------------------------
import os

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_IN100_DIR'] = DATA_DIR
os.environ['MSC_SCRATCH'] = MSC_ROOT

sess = M.Session(account='local', phase='p0', dataset='imagenet100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 worker_id=0, num_workers=1)

print()
print('layout under MSC_ROOT:')
print('  runs/{run_id}/  config.yaml  summary.json  STATUS.json')
print('                   metrics/     epochs.csv  final.csv  confusion_matrix.csv')
print('                                per_class.csv  exit_metrics.csv')
print('                   telemetry/   energy_samples.csv  system_samples.csv')
print('                                step_traces.jsonl')
print('                   per_sample/  test.parquet  train_holdout.parquet')
print('                                train_dynamics.parquet  meta.json')
print('                   checkpoints/ ckpt_last.pt  ckpt_best.pt')
print('                   env/         environment.json')
print('                   exit_heads.pt')
print('  budgets/{arch}.json     FLOPs per compute configuration')
print('  registry/events/*.jsonl  what ran, when, and how it ended')
print('  analysis/                Q1-Q4 outputs')
print('  tables/  paper/figures/  console/')

---
## Step 2 · Prove the pipeline is offline

Environment variables are a *request*. This blocks the socket layer outright,
so anything reaching for the network raises naming the host it wanted.

Installing a package is not offline-readiness, the same way draining an upload
queue is not confirmation that files landed.

In [ ]:
# Blocks outbound sockets (loopback stays open -- CUDA and the dataloader use
# it) and then builds every architecture and prices every budget table.
import msc_lib as M

fails = []
with M.no_network():
    for arch in M.zoo_for_dataset('imagenet100'):
        try:
            m = M.build_model(arch, dataset='imagenet100').eval()
            M.build_budget_table(arch, 'imagenet100', model=m)
            print(f'  [OK]   {arch}')
        except Exception as e:
            fails.append(f'{arch}: {type(e).__name__}: {e}')
            print(f'  [FAIL] {arch}: {fails[-1]}')

print()
if fails:
    print(f'{len(fails)} architecture(s) need the network or do not build.')
    print('The pipeline is NOT self-contained. Do not continue.')
else:
    print('OK -- all eight build and price with the network blocked.')

---
## Step 3 · Pack the dataset

129,395 loose JPEGs → one `256×256` uint8 memmap, with the splits frozen.
**~20–40 minutes on 24 cores, one time.**

You do not have to leave the notebook. The cell below runs the packer as a
subprocess with live output, using the `DATA_DIR` cell 2 already chose. Set
`SRC_DIR` to the folder that contains `train/`, and flip `RUN_PACKER` to True.

### How the labels are assigned

Folders sorted by WNID → class index 0…99. `n01440764` is 0, `n01855672` is 99.
Arbitrary but **deterministic and published**: the mapping lives in
`manifest.json` as `classes` and `class_names`. Nothing downstream depends on
matching official ImageNet indices, so an arbitrary-but-fixed labelling is
exactly right — what would matter is if it changed between runs, and the
fingerprint makes that impossible.

### The fingerprint is the thing to read

```
2b6269ef51ff87b2c9e00fa17c44326ce634a67892c9eb550ec518a6dd2d2b6c
```

A pure function of the file names and the split seeds, so it reproduces on any
machine. **A different value means your source tree differs from the one this
port was designed against.** It goes into `config_hash`, so two runs that
disagree about which 10,000 images are `val` refuse to be compared rather than
producing per-sample tables that align by index and describe different pictures.

Resumable at chunk granularity — an interruption continues rather than
restarting.

In [ ]:
# Set this to the folder that CONTAINS train/. Leave None to auto-detect.
SRC_DIR    = None
RUN_PACKER = False        # <-- flip to True to actually pack
#
# ONE TIME. Leaving it True afterwards is harmless: the cell checks whether the
# pack already exists and prints 'already packed -- nothing to do' without
# touching it. The packer itself is also idempotent and resumable, so nothing
# is redone and nothing is overwritten.

import subprocess, sys
from pathlib import Path

if SRC_DIR is None:
    # Look for a directory holding train/<wnid>/*.JPEG near the usual places.
    cands = [Path.home() / 'Desktop' / 'New folder',
             Path.cwd().parent, Path.cwd().parent.parent]
    for c in cands:
        try:
            t = Path(c) / 'train'
            if t.is_dir() and any(d.name.startswith('n0') for d in t.iterdir()):
                SRC_DIR = str(c)
                break
        except Exception:
            continue

ok, detail = M.data_present('imagenet100', DATA_DIR)
if ok:
    print('already packed -- nothing to do')
    print(' ', detail)
elif not RUN_PACKER:
    print('NOT PACKED YET. This is the next step, not a failure.')
    print()
    print(f'  source detected : {SRC_DIR or "NOT FOUND -- set SRC_DIR above"}')
    print(f'  destination     : {DATA_DIR}')
    print()
    print('  Set RUN_PACKER = True and re-run this cell (~20-40 min),')
    print('  or run the same thing in a terminal:')
    print(f'    python tools/pack_imagenet100.py --src "{SRC_DIR}" --out "{DATA_DIR}"')
elif not SRC_DIR:
    print('*** SRC_DIR not found. Set it to the folder containing train/.')
else:
    tool = Path.cwd().parent / 'tools' / 'pack_imagenet100.py'
    if not tool.exists():
        tool = Path.cwd() / 'tools' / 'pack_imagenet100.py'
    print(f'packing {SRC_DIR} -> {DATA_DIR}')
    print('(live output below; safe to interrupt, it resumes)')
    proc = subprocess.Popen([sys.executable, str(tool), '--src', str(SRC_DIR),
                             '--out', str(DATA_DIR)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line.rstrip())
    proc.wait()
    print(f'packer exit code {proc.returncode}')

In [ ]:
# Whatever route you took, verify the result here.
import json
from pathlib import Path

ok, detail = M.data_present('imagenet100', DATA_DIR)
print(f'pack present: {ok}')
print(f'  {detail}')
PACKED = ok

if ok:
    man = json.loads((Path(DATA_DIR) / 'manifest.json').read_text())
    spl = json.loads((Path(DATA_DIR) / 'splits.json').read_text())
    print()
    print(f"  images        {man['count']:,}")
    print(f"  classes       {man['n_classes']}   "
          f"({man['classes'][0]} = 0  ...  {man['classes'][-1]} = "
          f"{man['n_classes']-1})")
    print(f"  stored at     {man['stored_res']}px")
    print(f"  policy        {man['resize_policy']}")
    print(f"  val           {len(spl['val']):,}")
    print(f"  train         {len(spl['train']):,}")
    print(f"  holdout       {len(spl['holdout']):,}  (a slice OF train, aug off)")
    print(f"  decode fails  {len(man.get('failures', []))}  "
          f"(packed as zeros, excluded from every split)")
    print()
    print(f"  fingerprint   {man['fingerprint']}")
    EXPECTED = '2b6269ef51ff87b2c9e00fa17c44326ce634a67892c9eb550ec518a6dd2d2b6c'
    if man['fingerprint'] != EXPECTED:
        print()
        print('  *** FINGERPRINT DOES NOT MATCH THE DESIGN.')
        print(f'  *** expected {EXPECTED}')
        print('  *** Your source tree differs. Every downstream comparison')
        print('  *** would be against different images. Investigate first.')
else:
    print()
    print('  Not packed yet. Everything below still runs -- the preflight and')
    print('  both dry runs are SYNTHETIC and never open the dataset. Only')
    print('  NB2 onward needs it.')

---
## Step 4 · Preflight every architecture

Builds all eight, runs a forward and a backward, attaches an exit head to every
stage, and — the important part — **runs each one natively at every resolution
the oracle will sweep**: 96, 128, 160, 192, 224.

### Expect `swin_tiny` to fail at 96px, and possibly 128px

That is a recorded fact about the architecture, not a bug. Swin-T reduces its
input by 32 (patch-4 then three merges), so its final stage is 7×7 at 224 and
**3×3 at 96 — smaller than its own 7×7 attention window**.

The budget table probes per resolution and falls back to an analytic cost model
for the values that fail, and the *proxy* sweep (downsample-then-upsample) is
primary for all eight architectures anyway. What would be a real problem is the
failure going unrecorded.

### What each check prevents

| check | if it fails |
|---|---|
| builds / forwards / backprops | a torchvision decomposition that mis-ordered blocks |
| exit head attaches at every stage | a feature-rank or memory-layout mismatch (Swin speaks NHWC internally) |
| depth ρ strictly ascending, distinct, ends at 1.0 | **MSC is undefined when two budgets cost the same** |
| native resolution per value | a shape error discovered mid-sweep, hours in |

In [ ]:
report = M.preflight(sess, archs=M.zoo_for_dataset('imagenet100'), quick=False)
summ = M.preflight_summary(report)
failed = summ['failed']

print()
print(f"{len(summ['passed'])} passed, {len(failed)} failed, "
      f"{len(summ['todo'])} not yet done, of {summ['n']}")
for k in summ['todo']:
    print(f'  TODO   {k}  -- a prerequisite, not a failure')
for k in failed:
    print(f'  FAILED {k}  -- {report["checks"][k]["detail"]}')

# D-45: one profiler for the whole zoo, or cross-architecture rho is invalid.
used = M.profilers_used()
print()
print(f'FLOPs profiler: {sorted(used)}')
if len(used) > 1:
    print('  *** MORE THAN ONE PROFILER PRICED THIS ZOO. Cross-architecture')
    print('  *** transfer numbers would be invalid: the analytic fallback')
    print('  *** counts Conv2d and Linear only, so a transformer loses its')
    print('  *** attention matmuls entirely, and rho is DEFINED in FLOPs.')
    failed = failed + ['mixed FLOPs profilers']
else:
    print('  one profiler for all eight -- rho is comparable across the zoo')

---
## Step 5 · Dry-run the whole path, on synthetic data

Before any real training, push one synthetic batch through **the entire path
including evaluation and the artifact write**.

This is here because on CIFAR two defects each cost an hour of GPU time and each
was findable in milliseconds — but at *different* stages. One was the first
training step; the other was the history write at the **end** of epoch 0. A dry
run that stops after `loss.backward()` catches one and moves the other somewhere
else to hide.

So the backbone dry run goes: forward → loss → backward → optimiser step →
`optimisation_health` → `evaluate()` → history row (strict) → checkpoint save →
**checkpoint reload**. And the oracle dry run goes: every axis at every
resolution and precision → difficulty battery → prediction depth → per-sample
frame → parquet write → **parquet read back** → `compute_msc`.

In [ ]:
bad = []
# require_data=False: these are SYNTHETIC. They never open the dataset, so
# they must not wait on a 40-minute packing job to tell you a config is broken.
for arch in M.zoo_for_dataset('imagenet100'):
    cfg = sess.config(arch, seed=1, require_data=False)
    for name, fn in (('backbone', M.backbone_dry_run), ('oracle', M.oracle_dry_run)):
        ok, why = fn(cfg)
        print(f'  [{"OK" if ok else "FAIL"}] {arch:16s} {name:9s} {why}')
        if not ok:
            bad.append(f'{arch}/{name}: {why}')
print()
print('all dry runs pass' if not bad else f'{len(bad)} FAILED -- do not train')

---
## Step 6 · Kill-and-resume

**Do not skip this.** Five separate defects in the CIFAR programme were about
resume, and the worst of them silently restarted nine completed runs from epoch
0 — roughly 30 GPU-hours, with no error and no warning, because a re-trained run
looks exactly like normal work.

The test fires a **real interrupt** mid-run and then resumes in a fresh call. It
compares **per-epoch training loss after the seam**, not final accuracy: final
accuracy can match by luck, the loss curve cannot. An earlier version of this
test simulated a kill by training a shorter run and asking for more epochs —
that is a clean completion followed by an extension, a completely different code
path, and it validated nothing while looking like it did.

In [ ]:
res = M.resume_acceptance_test(sess, arch='resnet18', epochs=4, kill_at=2)
print()
print('RESUME OK' if res.get('passed') else 'RESUME FAILED -- do not train')
for k, v in res.items():
    if k != 'passed':
        print(f'  {k}: {v}')

---
## GO / NO-GO

In [ ]:
checks = {
    'offline (all 8 build with sockets blocked)': not fails,
    'dataset packed and fingerprint matches':     ok,
    'preflight (every arch, every resolution)':   not failed,
    'dry runs (train + measure paths)':           not bad,
    'kill-and-resume equivalence':                bool(res.get('passed')),
}
print()
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print()
if all(checks.values()):
    print('  GO -- open NB2_Train and run Phase 0 (4 runs, ~33 GPU-h).')
    print()
    print('  Phase 0 is 8% of the programme and it answers the only question')
    print('  that matters before committing the other 92%: does the ViT/CNN')
    print('  seed-reliability gap survive at ImageNet scale at all?')
else:
    print('  NO-GO. Fix the failures above. Training now would produce numbers')
    print('  that are well-formed and meaningless.')